# Producer Pattern 4 of 5 — Content-Based Router
### Beginner → real-world design → reusable implementation → deterministic testing

**Core idea:** One application receives different kinds of business events. A **content-based router** examines each event and chooses the correct Kafka topic *before* handing the message to the shared producer.

> **One sentence:** `validate event → decide topic and key → serialize → shared_producer.produce() → delivery callback`.

**Where this fits in your Kafka producer roadmap**

| Notebook | Pattern | What you learn |
|---|---|---|
| 1 | Fire-and-forget | Submit messages asynchronously |
| 2 | Callback-confirmed | Know whether Kafka acknowledged delivery |
| 3 | Singleton / shared producer | Reuse one producer across application services |
| **4 (this notebook)** | **Content-based router** | **Choose the destination topic from event content** |
| 5 (next) | Resilience / DLQ | Handle temporary outages, delivery failures, and failed records |

This tutorial uses **Python** and **`confluent-kafka`**. Every section states the goal, *why* the design exists, and *what to remember*. The tests work **without running Kafka**; live exercises are explicitly opt-in.

## Learning map

| Section | Objective | What an engineer retains |
|---|---|---|
| 0 | Set up a safe local lab | Do not delete topics in application code |
| 1 | Understand routing with an analogy | Router decides destination; producer sends |
| 2 | Start with `if/elif` | Routing is ordinary Python business logic |
| 3 | Define a consistent event envelope | Structured events are easier to route and observe |
| 4 | Validate before publishing | Reject malformed or unknown events early |
| 5 | Build a pure routing function | Routing must be testable without Kafka |
| 6 | Topic versus partition key | Topic selection is not partition selection |
| 7 | Connect router to shared producer | One producer sends to several topics |
| 8 | Delivery tracking and local queue pressure | Routing success ≠ Kafka delivery |
| 9 | Professional reusable implementation | Small responsibilities and explicit contracts |
| 10 | Broker-free tests | Prove routing and failure handling deterministically |
| 11 | Optional live Kafka demonstration | Observe real routed records |
| 12 | FastAPI lifecycle and deployment | Create once, reuse, drain on shutdown |
| 13 | Pitfalls and operational decisions | Prevent topic sprawl and silent misrouting |
| 14 | One-page engineer cheat sheet | Remember principles, not 500 lines of code |
| 15 | Next: resilience / DLQ | Know what this pattern does *not* solve |

---
# Section 0 — Safe notebook setup

**Objective:** Run the pure Python routing examples immediately, even if Kafka is not installed or running.

**Two execution modes:**

- **Offline (default):** routing exercises + simulated Kafka tests. No broker is needed.
- **Live (opt-in):** publish through a real Kafka producer. Requires `pip install confluent-kafka` and an accessible broker.

For a live run in a local environment, install the dependency in a separate notebook cell:

```python
%pip install confluent-kafka
```

Then set `RUN_LIVE_KAFKA = True` below. A unique *lab-only* topic prefix avoids touching your original exercises. No topic is deleted or recreated.

In [ ]:
import json
import logging
import os
import threading
import time
import uuid
from collections import deque
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Callable

# Keep this False to complete the offline learning path.
RUN_LIVE_KAFKA = False
BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")

# A unique lab prefix avoids colliding with your existing topics.
# For example: "lab4.orders", "lab4.payments", "lab4.payments.failed".
TOPIC_PREFIX = "lab4"

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("content_based_router")

print("Mode:", "LIVE Kafka" if RUN_LIVE_KAFKA else "OFFLINE tutorial")
print("Broker:", BOOTSTRAP)

---
# Section 1 — Why is a content-based router needed?

**Objective:** Explain the pattern in one minute, without Kafka internals.

## The sorting-desk analogy

Imagine an online store's mailroom. Parcels arrive from different departments:

- An **order-created** notice belongs in the **orders department**.
- A **successful payment** belongs in the **payments department**.
- A **failed payment** belongs in the **payment failures department**.
- A **low-stock notification** belongs in the **inventory alerts department**.

A **sorting clerk** examines the label/content and decides which department receives each parcel. A separate **courier service** performs transportation.

```text
                    BUSINESS EVENTS
                          |
                          v
                 +------------------+
                 | Content Router   |   <-- decides WHICH topic
                 +------------------+
                   /      |      \
                  /       |       \
             orders    payments   inventory
                 \        |       /
                  \       |      /
                   v       v     v
               +------------------+
               | Shared Producer  |   <-- sends the Kafka records
               +------------------+
                          |
                          v
                      KAFKA BROKERS
```

The **router does not send packets, manage broker connections, or confirm delivery**. It chooses a destination. The **shared producer** handles asynchronous Kafka publishing.

**Key distinction:** A *topic* is a named stream. A *partition* is a subdivision within a topic. Routing chooses the topic; Kafka uses the key/partitioner to determine the partition.

---
# Section 2 — Your first content-based router: plain `if/elif`

**Objective:** See that routing starts as an ordinary Python decision.

Let's deliberately write the smallest possible version. This is a teaching step, **not** yet our production implementation.

In [ ]:
def choose_topic_simple(event: dict) -> str:
    """Choose a topic based on the event's content."""
    event_type = event.get("event_type")
    data = event.get("data", {})

    if event_type == "order.created":
        return "lab4.orders"

    if event_type == "payment.updated" and data.get("status") == "FAILED":
        return "lab4.payments.failed"

    if event_type == "payment.updated":
        return "lab4.payments"

    if event_type == "inventory.changed" and data.get("available_units", 0) <= 5:
        return "lab4.inventory.alerts"

    if event_type == "inventory.changed":
        return "lab4.inventory"

    raise ValueError(f"Unknown event type: {event_type!r}")

examples = [
    {"event_type": "order.created", "data": {"order_id": "ORD-1"}},
    {"event_type": "payment.updated", "data": {"status": "FAILED"}},
    {"event_type": "inventory.changed", "data": {"available_units": 2}},
]
for example in examples:
    print(example["event_type"], "->", choose_topic_simple(example))

### Why do we need more than `if/elif`?

The basic example contains hidden risks:

1. Missing values might silently choose the wrong destination (look at `.get("available_units", 0)`).
2. The topic name, event fields, and routing decision are mixed into one function.
3. If an event is malformed, we should reject it **before** publishing.
4. Testability becomes harder when Kafka API calls are embedded inside every branch.

We will separate **event validation**, **topic selection**, and **message publishing**. This is not about adding unnecessary layers; it prevents routing bugs.

---
# Section 3 — Give every event a common envelope

**Objective:** Make events understandable to humans and predictable to software.

A consistent **event envelope** provides the same outer fields for every event:

| Field | Meaning |
|---|---|
| `event_id` | Unique identifier for tracking/deduplication |
| `event_type` | The business event kind |
| `occurred_at` | UTC timestamp when the event was created |
| `data` | The event-specific business information |

The envelope is not a Kafka requirement. It's our **application contract**.

In [ ]:
def make_event(event_type: str, data: dict, *, event_id: str | None = None) -> dict:
    """Wrap domain data in a consistent, JSON-friendly event envelope."""
    return {
        "event_id": event_id or str(uuid.uuid4()),
        "event_type": event_type,
        "occurred_at": datetime.now(timezone.utc).isoformat(),
        "data": data,
    }

order_event = make_event("order.created", {
    "order_id": "ORD-101",
    "customer_id": "CUST-7",
})
payment_failed_event = make_event("payment.updated", {
    "payment_id": "PAY-101",
    "order_id": "ORD-101",
    "status": "FAILED",
})
inventory_event = make_event("inventory.changed", {
    "sku": "LAPTOP-15",
    "available_units": 2,
    "reorder_point": 5,
})

print(json.dumps(order_event, indent=2))

**Professional note:** `event_id` is useful for correlating logs and handling duplicates downstream. It does **not** magically provide exactly-once delivery. A downstream system still needs an idempotency policy when duplicates matter.

---
# Section 4 — Validate business events before routing

**Objective:** Reject bad input early and clearly, rather than publishing an invalid event to an unexpected topic.

**Fail-fast policy:** If an event type or required field is invalid, raise an explicit exception. We do **not** silently route unknown events to another topic.

A production deployment usually uses an agreed event schema (e.g., JSON Schema, Avro, or Protobuf), versioning, and contract tests. Here we deliberately start with understandable Python checks.

In [ ]:
class EventValidationError(ValueError):
    """An event's structure or business fields are invalid."""


class UnknownEventTypeError(EventValidationError):
    """There is no approved routing rule for this event type."""


def require_nonempty_string(value: Any, field_name: str) -> str:
    if not isinstance(value, str) or not value.strip():
        raise EventValidationError(f"{field_name} must be a non-empty string")
    return value


def validate_event(event: dict) -> None:
    """Validate our tutorial's supported event contracts."""
    if not isinstance(event, dict):
        raise EventValidationError("event must be a dictionary")

    for name in ("event_id", "event_type", "occurred_at"):
        require_nonempty_string(event.get(name), name)

    data = event.get("data")
    if not isinstance(data, dict):
        raise EventValidationError("data must be a dictionary")

    event_type = event["event_type"]

    if event_type == "order.created":
        require_nonempty_string(data.get("order_id"), "data.order_id")
        require_nonempty_string(data.get("customer_id"), "data.customer_id")

    elif event_type == "payment.updated":
        require_nonempty_string(data.get("payment_id"), "data.payment_id")
        require_nonempty_string(data.get("order_id"), "data.order_id")
        if data.get("status") not in {"PENDING", "SUCCEEDED", "FAILED"}:
            raise EventValidationError("data.status must be PENDING, SUCCEEDED, or FAILED")

    elif event_type == "inventory.changed":
        require_nonempty_string(data.get("sku"), "data.sku")
        for field in ("available_units", "reorder_point"):
            number = data.get(field)
            if type(number) is not int or number < 0:
                raise EventValidationError(f"data.{field} must be a nonnegative integer")

    else:
        raise UnknownEventTypeError(f"No route defined for event_type={event_type!r}")


for event in (order_event, payment_failed_event, inventory_event):
    validate_event(event)
print("All three example events passed validation.")

In [ ]:
# Negative examples: incorrect event data should be rejected, not misrouted.
invalid_stock = make_event("inventory.changed", {
    "sku": "LAPTOP-15",
    "available_units": -2,
    "reorder_point": 5,
})
unknown_event = make_event("customer.favorite_color_changed", {"color": "blue"})

for event in (invalid_stock, unknown_event):
    try:
        validate_event(event)
    except EventValidationError as exc:
        print(type(exc).__name__ + ":", exc)

### Remember

An invalid event is **not** a Kafka delivery failure. It is a **business contract failure** detected locally, before `produce()` is called. The application can return an API validation error or send the event through an explicitly designed remediation workflow. Do not automatically dump every invalid event into a dead-letter topic.

---
# Section 5 — Build a pure routing decision function

**Objective:** Return an explicit `topic`, `message key`, and `reason` without touching Kafka.

A **pure routing function** receives an event and returns a decision. Because it has no network calls or mutable state, you can test it with ordinary Python assertions.

**Rules for this tutorial:**

| Event content | Destination |
|---|---|
| `order.created` | `lab4.orders` |
| `payment.updated` + `status=FAILED` | `lab4.payments.failed` |
| Other valid `payment.updated` | `lab4.payments` |
| `inventory.changed` + `available_units <= reorder_point` | `lab4.inventory.alerts` |
| Other valid `inventory.changed` | `lab4.inventory` |

Topic names are **allowlisted** in code. Do not let untrusted users supply arbitrary topic names.

In [ ]:
@dataclass(frozen=True)
class RouteDecision:
    """An immutable decision describing where a message belongs."""
    topic: str
    key: str
    reason: str


def route_event(event: dict, *, prefix: str = TOPIC_PREFIX) -> RouteDecision:
    """Validate an event, then choose a topic and partition key."""
    validate_event(event)
    event_type = event["event_type"]
    data = event["data"]

    if event_type == "order.created":
        return RouteDecision(
            topic=f"{prefix}.orders",
            key=data["order_id"],
            reason="order_created",
        )

    if event_type == "payment.updated":
        if data["status"] == "FAILED":
            return RouteDecision(
                topic=f"{prefix}.payments.failed",
                key=data["order_id"],
                reason="payment_failed",
            )
        return RouteDecision(
            topic=f"{prefix}.payments",
            key=data["order_id"],
            reason="payment_non_failed",
        )

    # Reaching here means event_type == inventory.changed.
    if data["available_units"] <= data["reorder_point"]:
        return RouteDecision(
            topic=f"{prefix}.inventory.alerts",
            key=data["sku"],
            reason="at_or_below_reorder_point",
        )

    return RouteDecision(
        topic=f"{prefix}.inventory",
        key=data["sku"],
        reason="stock_healthy",
    )


for event in (order_event, payment_failed_event, inventory_event):
    decision = route_event(event)
    print(event["event_type"], "->", decision)

### Why return a `RouteDecision` object?

Without it, applications often scatter topic names across API handlers, payment services, inventory services, and tests. A small decision object makes *topic selection* observable, testable, and independent of Kafka.

**Scope note:** This router chooses **one destination topic per event**. Event fan-out (publishing the same record to several topics) introduces multiple independent deliveries and is a different requirement.

---
# Section 6 — Topic routing versus partitioning

**Objective:** Understand why we also return a message key.

- **Topic:** Which stream receives this event? (`lab4.orders` versus `lab4.payments.failed`).
- **Key:** Which logical entity should have stable partition affinity *within that topic*? (`ORD-101` or `LAPTOP-15`).
- **Partition:** Kafka's partitioner selects one of the topic's partitions, typically based on the key.

```text
Event: order.created, order_id=ORD-101
      |
      +--> Router chooses topic: lab4.orders
      |
      +--> Router chooses key:   ORD-101
                                    |
                                    v
                          Kafka partitioner
                                    |
                                    v
                        one partition in lab4.orders
```

**Why same key?** Related order events sent to the **same topic** can use the same partition, helping preserve their order *within that partition* when producer ordering guarantees apply.

**Important:** Kafka provides **no general ordering guarantee across different topics**. Even if two messages both use key `ORD-101`, `lab4.orders` and `lab4.payments.failed` are independent streams.

In [ ]:
# Routing is independent of Kafka and can be asserted directly.
normal_payment = make_event("payment.updated", {
    "payment_id": "PAY-200",
    "order_id": "ORD-101",
    "status": "SUCCEEDED",
})
healthy_inventory = make_event("inventory.changed", {
    "sku": "MOUSE-9", "available_units": 50, "reorder_point": 5
})

assert route_event(order_event).topic == "lab4.orders"
assert route_event(payment_failed_event).topic == "lab4.payments.failed"
assert route_event(normal_payment).topic == "lab4.payments"
assert route_event(inventory_event).topic == "lab4.inventory.alerts"
assert route_event(healthy_inventory).topic == "lab4.inventory"
assert route_event(order_event).key == "ORD-101"
assert route_event(payment_failed_event).key == "ORD-101"
print("All offline routing rules passed.")

---
# Section 7 — Connect the router to the *shared* Kafka producer

**Objective:** See the end-to-end relationship with Notebook 3.

The router makes the decision. The **application-owned, reusable** Kafka producer publishes the chosen record. We do **not** construct a new `Producer` for every event or every topic.

```text
HTTP order endpoint ---\
Payment service --------> route_event(event) --> shared_producer.produce(topic, key, value)
Inventory service -----/                             |
                                                     v
                                               Kafka brokers
```

A successful `produce()` call means **local queue acceptance**, not broker delivery. Notebook 2 taught us that a delivery callback provides the eventual success/failure result.

We first build an adapter against a *fake* producer so you can learn the integration without Kafka.

In [ ]:
def serialize_event(event: dict) -> bytes:
    """Convert a validated event into Kafka's byte payload."""
    try:
        return json.dumps(
            event,
            ensure_ascii=False,
            allow_nan=False,
            separators=(",", ":"),
        ).encode("utf-8")
    except (TypeError, ValueError) as exc:
        raise EventValidationError(f"Event cannot be JSON-encoded: {exc}") from exc


# This intentionally uses a tiny print-only stand-in for Producer.
class PrintProducer:
    def produce(self, *, topic, key, value, on_delivery=None, headers=None):
        print("QUEUED:", topic, "key=", key.decode(), "bytes=", len(value))

    def poll(self, timeout):
        return 0


def publish_one_event(producer, event: dict) -> RouteDecision:
    """Minimal integration: route, serialize, submit."""
    decision = route_event(event)
    payload = serialize_event(event)

    producer.produce(
        topic=decision.topic,
        key=decision.key.encode("utf-8"),
        value=payload,
    )
    producer.poll(0)  # service any ready producer events
    return decision

shared_fake = PrintProducer()
for event in (order_event, payment_failed_event, inventory_event):
    publish_one_event(shared_fake, event)

**What changed from Notebook 3?** Only the **destination selection**. The producer is still shared, asynchronous, and responsible for sending bytes. The router should not know which TCP connections exist or how brokers implement acknowledgements.

---
# Section 8 — Delivery callbacks and backpressure still apply

**Objective:** Prevent two common misunderstandings.

### A. Routing success is NOT delivery success

The router saying `lab4.orders` is analogous to writing the correct address on an envelope. The courier still needs to transport it. Register `on_delivery` to observe the eventual Kafka delivery result.

### B. `BufferError` is a **local** queue-full condition

`produce()` can raise `BufferError` if the producer cannot accept another message. The record was **not accepted locally** in that attempt. It can be retried after a short wait, subject to a deadline. This is different from a delivery callback reporting a failure *after* local acceptance.

For production, think about **two retry policies**:

| Failure | Where | Handling |
|---|---|---|
| Local `BufferError` | `produce()` raises before accepting the event | Brief backpressure/retry with a deadline |
| Final delivery error | `on_delivery(error, message)` | Alert, persist failure, or invoke a durable recovery process (Notebook 5) |

Never automatically retry an already-accepted event without a deduplication strategy: the final outcome may be uncertain and duplicates are possible.

---
# Section 9 — Production-oriented `KafkaEventRouter`

**Objective:** Package the responsibilities into a small, reusable component while staying readable to a junior engineer.

### Design principles

1. **Dependency injection:** Accept an existing producer. The application owns its lifecycle.
2. **Pure decision first:** Validate and route before touching Kafka.
3. **One producer, many topics:** No producer-per-route or producer-per-request.
4. **Callback-confirmed metrics:** Track accepted, acknowledged, failed, and pending callback counts separately.
5. **Bounded local backpressure:** Do not retry `BufferError` indefinitely.
6. **No false delivery claims:** `publish()` returns *locally accepted*, not *delivered*.

The class deliberately does not implement broker-outage retry, persistence, DLQ, or an elaborate schema registry. Those are separate responsibilities, introduced in Notebook 5 or later.

In [ ]:
class LocalQueueTimeout(TimeoutError):
    """The producer's local queue remained full past our retry deadline."""


@dataclass(frozen=True)
class PublishReceipt:
    """Local acceptance receipt; this is NOT a broker acknowledgement."""
    event_id: str
    topic: str
    key: str
    route_reason: str


class KafkaEventRouter:
    """
    Validate, route, and publish business events using ONE shared producer.

    The application creates the underlying Producer once and injects it here.
    The application also calls producer.flush() at shutdown.
    """

    def __init__(
        self,
        producer: Any,
        *,
        topic_prefix: str = TOPIC_PREFIX,
        queue_wait_seconds: float = 3.0,
        log: logging.Logger | None = None,
    ) -> None:
        if queue_wait_seconds <= 0:
            raise ValueError("queue_wait_seconds must be positive")
        self._producer = producer
        self._topic_prefix = topic_prefix
        self._queue_wait_seconds = queue_wait_seconds
        self._logger = log or logging.getLogger("KafkaEventRouter")
        self._lock = threading.Lock()  # metrics can be updated by different poll callers
        self._stats = {"accepted": 0, "delivered": 0, "failed": 0}

    def publish(self, event: dict) -> PublishReceipt:
        """
        Route the event and queue it for asynchronous Kafka delivery.

        Returns only after produce() accepts the record locally.
        Delivery success/failure arrives later through a callback.
        """
        decision = route_event(event, prefix=self._topic_prefix)
        payload = serialize_event(event)
        event_id = event["event_id"]

        def on_delivery(error, message) -> None:
            # Keep callbacks lightweight: count and log, not slow network calls.
            with self._lock:
                if error is None:
                    self._stats["delivered"] += 1
                else:
                    self._stats["failed"] += 1

            if error is not None:
                self._logger.error(
                    "Kafka delivery failed event_id=%s topic=%s error=%s",
                    event_id, decision.topic, error,
                )

        deadline = time.monotonic() + self._queue_wait_seconds

        while True:
            try:
                self._producer.produce(
                    topic=decision.topic,
                    key=decision.key.encode("utf-8"),
                    value=payload,
                    # Headers are for downstream consumers; don't expect them
                    # to be returned in confluent-kafka's delivery callback.
                    headers=[
                        ("event_id", event_id.encode("utf-8")),
                        ("event_type", event["event_type"].encode("utf-8")),
                    ],
                    on_delivery=on_delivery,
                )
                break  # Accepted into the local queue, not yet delivered.

            except BufferError as exc:
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    raise LocalQueueTimeout(
                        f"Local Kafka queue stayed full for event_id={event_id}"
                    ) from exc
                # Service events, giving the producer a chance to progress.
                self._producer.poll(min(0.2, remaining))

        with self._lock:
            self._stats["accepted"] += 1

        # Don't block waiting for broker delivery during normal publishing.
        self._producer.poll(0)

        return PublishReceipt(
            event_id=event_id,
            topic=decision.topic,
            key=decision.key,
            route_reason=decision.reason,
        )

    def snapshot(self) -> dict:
        """Return a point-in-time copy of callback and acceptance counters."""
        with self._lock:
            counts = self._stats.copy()
        counts["awaiting_callbacks"] = (
            counts["accepted"] - counts["delivered"] - counts["failed"]
        )
        return counts


print("KafkaEventRouter is defined. It has not connected to Kafka yet.")

### Read the implementation as five small responsibilities

| Component | Why it exists |
|---|---|
| `route_event()` | Centralized, deterministic business routing |
| `serialize_event()` | Kafka sends bytes, not Python dictionaries |
| `producer.produce(...)` | Asynchronously queue the chosen record |
| `on_delivery(...)` | Capture final delivery outcomes |
| `producer.poll(0)` | Service ready producer callbacks during publishing |

**Why no `Producer(...)` in this class?** From Notebook 3, we learned that application startup creates the **shared** producer and injects it into dependent components. That is the practical dependency-injection pattern.

**Concurrency detail:** The producer supports use from multiple threads. Metrics have a lock because callbacks can be served by whichever application thread calls `poll()` or `flush()`. A snapshot taken during concurrent submission can briefly be out of sync; final settled counts are meaningful.

**Lifecycle ownership:** The wrapper does not shut down a shared producer itself. Doing so would break other services that still use it. The application calls `flush()` *once* during a coordinated shutdown after it has stopped admitting new work.

---
# Section 10 — Offline tests: prove behavior without Kafka

**Objective:** Test *your routing and publishing decisions* deterministically. A live broker is unnecessary for these unit tests.

A **fake producer** records calls to `produce()` and simulates callback execution during `poll()`/`flush()`. It does **not** simulate Kafka's network, partitioner, or actual broker durability.

In [ ]:
class FakeKafkaMessage:
    """Only the Message API surface needed for our callback tests."""
    def __init__(self, topic: str, partition: int, offset: int):
        self._topic, self._partition, self._offset = topic, partition, offset

    def topic(self): return self._topic
    def partition(self): return self._partition
    def offset(self): return self._offset


class FakeProducer:
    """A deterministic producer stand-in; does not contact a Kafka broker."""

    def __init__(self, *, fail_event_ids=(), full_attempts=0, always_full=False):
        self.calls = []
        self.pending = deque()
        self.fail_event_ids = set(fail_event_ids)
        self.full_attempts = full_attempts
        self.always_full = always_full
        self.poll_calls = 0
        self.flush_calls = 0
        self.offset = 0

    def produce(self, *, topic, key, value, on_delivery=None, headers=None):
        if self.always_full or self.full_attempts > 0:
            if not self.always_full:
                self.full_attempts -= 1
            raise BufferError("simulated local queue full")

        record = {"topic": topic, "key": key, "value": value, "headers": headers}
        self.calls.append(record)
        self.pending.append((record, on_delivery))

    def _complete_one(self):
        if not self.pending:
            return 0
        record, callback = self.pending.popleft()
        parsed = json.loads(record["value"])
        error = (RuntimeError("simulated broker delivery failure")
                 if parsed["event_id"] in self.fail_event_ids else None)
        if callback is not None:
            callback(error, FakeKafkaMessage(record["topic"], 0, self.offset))
        self.offset += 1
        return 1

    def poll(self, timeout):
        self.poll_calls += 1
        if not self.pending and timeout > 0:
            time.sleep(min(timeout, 0.002))
        # One event per poll is a test convention, not Kafka's exact behavior.
        return self._complete_one()

    def flush(self, timeout):
        self.flush_calls += 1
        while self.pending:
            self._complete_one()
        return 0


print("Fake producer ready: no Kafka broker required.")

### Test A — The right destination and key are chosen

This verifies the actual business rule: payment failures go to a different topic than successful payments, while order and inventory events follow their own destinations.

In [ ]:
fake = FakeProducer()
router = KafkaEventRouter(fake)

receipts = [router.publish(event) for event in (
    order_event, payment_failed_event, normal_payment, inventory_event, healthy_inventory
)]

expected_topics = [
    "lab4.orders", "lab4.payments.failed", "lab4.payments",
    "lab4.inventory.alerts", "lab4.inventory",
]
assert [r.topic for r in receipts] == expected_topics
assert [call["topic"] for call in fake.calls] == expected_topics
assert fake.calls[0]["key"] == b"ORD-101"
assert fake.calls[1]["key"] == b"ORD-101"
assert fake.calls[3]["key"] == b"LAPTOP-15"

assert fake.flush(5) == 0
assert router.snapshot() == {"accepted": 5, "delivered": 5, "failed": 0, "awaiting_callbacks": 0}
print("PASS: Five events reached five expected destinations using one producer.")

### Test B — Invalid and unknown events never reach the producer

Failing before a Kafka call is a design feature. A malformed event should not be published into a topic just because a field was missing.

In [ ]:
fake = FakeProducer()
router = KafkaEventRouter(fake)

for bad_event in (invalid_stock, unknown_event):
    try:
        router.publish(bad_event)
        raise AssertionError("Expected an EventValidationError")
    except EventValidationError:
        pass

assert len(fake.calls) == 0
print("PASS: Invalid/unknown events were rejected before any produce() call.")

### Test C — Kafka delivery failure is different from routing failure

We deliberately simulate a Kafka failure **after** the event was accepted locally. The routing receipt still exists, but the delivery callback reports failure.

In [ ]:
failed_id = "event-that-will-fail"
failing_event = make_event(
    "order.created",
    {"order_id": "ORD-FAIL", "customer_id": "CUST-1"},
    event_id=failed_id,
)
fake = FakeProducer(fail_event_ids={failed_id})
router = KafkaEventRouter(fake)
receipt = router.publish(failing_event)

assert receipt.topic == "lab4.orders"   # routing was correct
assert fake.flush(5) == 0                 # callback processing finished
assert router.snapshot()["failed"] == 1
assert router.snapshot()["delivered"] == 0
print("PASS: Routing succeeded, but broker delivery was reported as failed.")

### Test D — Recover from temporary local queue pressure

The first two calls to `produce()` will raise `BufferError`. The router polls and retries **the same event**, with a deadline.

In [ ]:
fake = FakeProducer(full_attempts=2)
router = KafkaEventRouter(fake, queue_wait_seconds=0.5)
receipt = router.publish(order_event)
assert receipt.topic == "lab4.orders"
assert len(fake.calls) == 1  # exactly one accepted record, not three
assert fake.poll_calls >= 2
assert fake.flush(5) == 0
assert router.snapshot()["delivered"] == 1
print("PASS: Temporary queue pressure was retried without duplicate local acceptance.")

### Test E — Permanent queue pressure fails within a deadline

Never build an unbounded `while True` retry loop that can hang a request forever if Kafka cannot keep up.

In [ ]:
fake = FakeProducer(always_full=True)
router = KafkaEventRouter(fake, queue_wait_seconds=0.04)
started = time.monotonic()

try:
    router.publish(order_event)
    raise AssertionError("Expected LocalQueueTimeout")
except LocalQueueTimeout:
    pass

assert time.monotonic() - started < 1.0
assert len(fake.calls) == 0
print("PASS: Permanent queue pressure failed within a bounded time.")

### Test F — Services share one producer instance

This tests the integration lesson from Notebook 3: multiple business services should not construct separate Kafka clients when the same client configuration suffices.

In [ ]:
fake_shared = FakeProducer()
shared_router = KafkaEventRouter(fake_shared)

def order_service_create(event):
    return shared_router.publish(event)

def payment_service_update(event):
    return shared_router.publish(event)

def inventory_service_update(event):
    return shared_router.publish(event)

order_service_create(order_event)
payment_service_update(payment_failed_event)
inventory_service_update(inventory_event)
assert len(fake_shared.calls) == 3
assert len({id(shared_router._producer) for _ in range(3)}) == 1
fake_shared.flush(5)
assert shared_router.snapshot()["delivered"] == 3
print("PASS: Three services routed three events using one shared producer.")

### Test G — Callback handling under concurrent API requests

Thread safety matters because a web server may have several threads using the same producer. The test checks our router's counters and event destinations. It does not benchmark Kafka throughput.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

fake = FakeProducer()
router = KafkaEventRouter(fake)

def send_one(index):
    event = make_event("order.created", {
        "order_id": f"ORD-THREAD-{index}",
        "customer_id": "CUST-THREAD",
    })
    return router.publish(event)

with ThreadPoolExecutor(max_workers=6) as pool:
    receipts = list(pool.map(send_one, range(40)))

remaining = fake.flush(5)
counts = router.snapshot()
assert remaining == 0
assert len(receipts) == 40
assert counts["accepted"] == counts["delivered"] == 40
assert counts["failed"] == counts["awaiting_callbacks"] == 0
print("PASS: 40 concurrent calls routed correctly using one producer.")

### What the offline tests guarantee — and what they do not

They verify **your Python logic**: validation, routing decisions, keys, local retries, callback counters, and shared client injection.

They do **not** verify: broker availability, partitions, actual acknowledgements, consumer processing, Kafka cluster security, schema compatibility, or persistence durability. Those require integration tests and operational monitoring.

---
# Section 11 — Optional LIVE Kafka lab

**Objective:** Send the same five event types to real Kafka topics while maintaining **one producer**.

This section is disabled by default. To enable it:

1. Start your local Kafka broker (the earlier notebooks used `localhost:9092`).
2. Install `confluent-kafka` in this Python environment.
3. Change `RUN_LIVE_KAFKA = True` in Section 0 and rerun the notebook.

**Safety:** The helper below creates *only missing lab topics*. It never deletes or recreates an existing topic. In real deployments, topic provisioning belongs to infrastructure automation or administrators, not every application startup.

In [ ]:
if RUN_LIVE_KAFKA:
    from confluent_kafka.admin import AdminClient, NewTopic

    def ensure_live_topics(topic_names: list[str]) -> None:
        """Create absent lab topics; do not modify or delete existing topics."""
        admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
        existing = set(admin.list_topics(timeout=8).topics)
        missing = sorted(set(topic_names) - existing)
        if not missing:
            print("Lab topics already exist.")
            return

        futures = admin.create_topics([
            NewTopic(name, num_partitions=3, replication_factor=1)
            for name in missing
        ])
        for topic, future in futures.items():
            try:
                future.result(timeout=15)
                print("Created lab topic:", topic)
            except Exception as exc:
                # Report failures explicitly; don't assume a topic is ready.
                raise RuntimeError(f"Could not create lab topic {topic}: {exc}") from exc

    live_topics = [f"{TOPIC_PREFIX}.{suffix}" for suffix in (
        "orders", "payments", "payments.failed", "inventory", "inventory.alerts"
    )]
    ensure_live_topics(live_topics)
else:
    print("Offline mode: topic creation skipped.")

In [ ]:
if RUN_LIVE_KAFKA:
    from confluent_kafka import Producer

    # Exactly one Producer instance serves every routed topic in this lab.
    shared_producer = Producer({
        "bootstrap.servers": BOOTSTRAP,
        "client.id": "notebook-04-content-router",
        "acks": "all",
        "enable.idempotence": True,
        "delivery.timeout.ms": 30000,
    })
    live_router = KafkaEventRouter(shared_producer, topic_prefix=TOPIC_PREFIX)

    live_events = [
        make_event("order.created", {"order_id": "ORD-LIVE-1", "customer_id": "CUST-5"}),
        make_event("payment.updated", {"payment_id": "PAY-LIVE-1", "order_id": "ORD-LIVE-1", "status": "SUCCEEDED"}),
        make_event("payment.updated", {"payment_id": "PAY-LIVE-2", "order_id": "ORD-LIVE-2", "status": "FAILED"}),
        make_event("inventory.changed", {"sku": "SKU-LOW", "available_units": 1, "reorder_point": 5}),
        make_event("inventory.changed", {"sku": "SKU-HIGH", "available_units": 40, "reorder_point": 5}),
    ]

    try:
        for event in live_events:
            receipt = live_router.publish(event)
            print("LOCAL ACCEPTANCE:", receipt.event_id, "->", receipt.topic)
    finally:
        # Caller owns the shared producer lifecycle, not the router wrapper.
        remaining = shared_producer.flush(timeout=35)
        print("Flush remaining:", remaining)
        print("Delivery outcomes:", live_router.snapshot())

    if remaining or live_router.snapshot()["failed"]:
        print("WARNING: Check delivery failures or outstanding messages.")
else:
    print("Offline mode: live publishing skipped. Offline tests above are complete.")

### How to interpret the LIVE results

- `LOCAL ACCEPTANCE:` means the router chose a topic and `produce()` queued the record.
- `delivered` counts successful delivery callbacks according to the configured Kafka acknowledgement policy.
- `failed` counts final delivery failures reported by Kafka's producer client.
- `Flush remaining: 0` means no outstanding producer work remains, **not automatically that all messages succeeded**. Always inspect failure counts.

To inspect messages, use your local Kafka CLI or another consumer. Do **not** assume a delivery callback means a consumer processed the message.

---
# Section 12 — Where this belongs in a FastAPI application

**Objective:** Translate the notebook to a maintainable project structure.

```text
app/
├── main.py               # Creates and drains ONE shared Kafka producer
├── events.py             # Event definitions + validation
├── routing.py            # Pure route_event(event)
├── kafka_publisher.py    # KafkaEventRouter wrapper + callbacks
└── services/
    ├── orders.py         # Emits order.created
    ├── payments.py       # Emits payment.updated
    └── inventory.py      # Emits inventory.changed
```

**Conceptual FastAPI lifespan integration:**

```python
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request
from confluent_kafka import Producer

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup: one application-scoped producer per worker process.
    producer = Producer({"bootstrap.servers": BOOTSTRAP})
    app.state.producer = producer
    app.state.router = KafkaEventRouter(producer)

    yield  # Application handles requests using app.state.router.

    # Shutdown: stop accepting new requests, then drain pending work.
    outstanding = producer.flush(30)
    if outstanding:
        logger.error("Shutdown left %s Kafka events outstanding", outstanding)

app = FastAPI(lifespan=lifespan)
```

**Production considerations:**

- Each **worker process** gets its own producer instance; this is not a global singleton across machines or processes.
- `Producer` is thread-safe, but its synchronous `poll()` / `flush()` should not block a high-throughput `asyncio` event loop. For async applications, move blocking work to a managed thread or use a supported async producer design.
- Application shutdown should first stop new publishing and coordinate active requests, then flush with a deadline.
- The API response must not claim *"Kafka confirmed the event"* merely because `publish()` returned. It only proves the local client accepted the event.
- For strict DB + event consistency, see the **transactional outbox** pattern; routing alone cannot make a DB write and Kafka publish atomic.

---
# Section 13 — Production design review

**Objective:** Anticipate the mistakes that would matter in a real code review.

| Mistake | What breaks | Safer design |
|---|---|---|
| `topic = event["topic"]` from untrusted payload | Arbitrary writes, topic sprawl, security issues | Explicit allowlisted routing rules |
| Use `.get("status", "FAILED")` | Missing data is silently misrouted | Validate required fields first |
| New `Producer` for each destination | Too many connections, weaker batching, overhead | Reuse one producer for compatible configurations |
| Claim that `publish()` means delivered | Incorrect API/business guarantees | Call it *local acceptance* and monitor callbacks |
| Ignore delivery callback failures | Silent data loss | Alert and define durable recovery (Notebook 5) |
| Retry every accepted event blindly | Potential duplicate business events | Use event IDs, idempotent consumers, outbox/recovery design |
| Create a topic for every customer | High-cardinality topics and operational complexity | Finite set of topics + message keys / headers |
| Assume same key orders across topics | Cross-topic ordering bug | Only rely on guarantees within a partition |
| No schema/version strategy | Consumers break after payload changes | Explicit schema evolution and compatibility tests |
| Heavy work inside callbacks | Slow event servicing | Keep callback small, hand off recovery work safely |
| Call `flush()` for every event | Destroys batching and adds latency | Poll regularly; flush at coordinated shutdown |
| Router owns global producer shutdown | Breaks other publishers | Lifecycle belongs to application startup/shutdown |

### Routing policy questions a senior reviewer would ask

- What happens for **unknown** event types? (Here: reject.)
- Who owns changing the topic map? (Versioned application configuration/code review.)
- How are topic permissions provisioned? (Broker ACLs/service identity.)
- What guarantees are expected when payment updates move between topics? (No cross-topic atomicity/ordering.)
- What happens when a message is accepted locally but later fails delivery? (Explicit recovery design, next notebook.)
- How do you correlate API requests to Kafka results? (`event_id` + observability.)

### Minimal production metrics

`events_routed_total{event_type,topic}` · `events_rejected_total{reason}` · `kafka_delivery_success_total{topic}` · `kafka_delivery_failed_total{topic}` · `queue_full_retries_total` · `delivery_latency_ms`.

Keep metrics labels **low cardinality**: event IDs and customer IDs belong in trace/log context, not metric labels.

---
# Section 14 — What engineers actually remember

This is the portion to retain after closing the notebook.

## The six concepts worth memorizing

1. **A router chooses the topic from event content.** It does not send data itself.
2. **Make routing deterministic and testable.** `route_event(event) -> RouteDecision(topic, key, reason)`.
3. **Validate before routing.** Unknown or malformed events must not silently reach a random topic.
4. **Reuse a shared producer.** One Kafka client can publish asynchronously to many topics.
5. **Keep delivery separate from routing.** `produce()` queues locally; callbacks tell you eventual Kafka delivery outcomes.
6. **Know your boundaries.** Topic routing does not guarantee cross-topic ordering, exactly-once processing, or recovery after failure.

### The smallest mental model

```text
Business event
      |
      v
VALIDATE ------invalid------> reject / explicit remediation
      |
      v
ROUTE(event) -> {topic, key}
      |
      v
SHARED PRODUCER.produce(...)
      |
      v
BACKGROUND NETWORKING -> Kafka broker
      |
      v
DELIVERY CALLBACK via poll()/flush()
```

### One implementation pattern to remember

```python
# Application startup: create ONE shared producer.
producer = Producer({"bootstrap.servers": BOOTSTRAP})
router = KafkaEventRouter(producer)

# Anywhere in business logic:
receipt = router.publish(event)   # Topic chosen from validated content.
# receipt means LOCAL ACCEPTANCE, not confirmed delivery.

# Application shutdown (after new publishing stops):
remaining = producer.flush(30)
# Check outstanding count + delivery failures.
```

**Interview explanation (30 seconds):**

> "A content-based router examines validated business-event fields and selects an allowlisted Kafka topic and partition key. I keep that decision as a pure function, publish using a shared thread-safe producer, and observe delivery separately through callbacks. I reject unknown event types, avoid dynamic topic sprawl, and don't assume cross-topic ordering or atomic delivery."

---
# Section 15 — Exercises and the next notebook

## Practice exercises (in increasing difficulty)

1. Add `shipment.created` routed to `lab4.shipments` with `shipment_id` as key. Include validation and a unit test.
2. Add a new payment status `REFUNDED`, routing it to `lab4.payments.refunds`. Decide how you'll preserve schema compatibility.
3. Add `route_version` to your routing decision or application logs. Explain how you would roll out a new rule without mixing records unexpectedly.
4. Write a test proving two different inventory SKUs receive different keys, while repeated updates for one SKU use the same key.
5. Explain why a failed delivery callback must **not** directly trigger an uncontrolled retry loop in a real service.
6. Draw a sequence diagram of an HTTP request that queues an event, returns success, and later receives a delivery failure. What must the business design do?

## Notebook 5 — Resilient Producer / Dead Letter Queue

Next we'll address the unsolved problem:

**What happens when Kafka is unavailable or a message permanently fails?**

You will learn: bounded retries, retry classification, backoff, circuit breaking, durable recovery, and the difference between a true DLQ workflow versus dropping unsent data into an in-memory list.

The content router will remain the **destination decision** component. Resilience will be a separate layer, rather than hiding retry and DLQ logic inside every routing branch.

---
## Official references and further reading

1. [Confluent Kafka Python Client — Overview and asynchronous writes](https://docs.confluent.io/kafka-clients/python/current/overview.html)
2. [Confluent Kafka Python API — `Producer.produce()`, `poll()`, `flush()`](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html)
3. [Confluent Kafka Python Client — Thread-safety reference](https://github.com/confluentinc/confluent-kafka-python/blob/master/docs/multithreading-guide.md)
4. [Apache Kafka Documentation — Producer concepts and configuration](https://kafka.apache.org/documentation/)

**A careful distinction:** This notebook presents an **application-level content-based router design pattern** implemented with the Kafka Producer API. Kafka does not expose a special `ContentBasedRouter` producer class; we implement the selection logic ourselves.